# 아산시 방송 홍보 효과 분석 - Step 0: 크롤링 + 즉시 분석
> 지금 있는 데이터(크롤링 API + T맵)로 바로 돌리는 버전

**수집 대상:**
1. 네이버 데이터랩 - 검색 트렌드 (방송 전후 비교)
2. 네이버 블로그/뉴스 - 콘텐츠 수 변화
3. 유튜브 - 영상 업로드/조회수
4. T맵 - 관광지 방문 변화

In [ ]:
# 패키지 설치
!pip install httpx fake-useragent tenacity python-dotenv beautifulsoup4 pandas matplotlib tqdm -q

In [ ]:
import os, sys, json, re, time, glob
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rc
import httpx
from dotenv import load_dotenv

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
try:
    font_path = 'C:/Windows/Fonts/malgun.ttf'
    rc('font', family=font_manager.FontProperties(fname=font_path).get_name())
except:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Ready')

In [ ]:
# ============================
# 경로 & API 키 로드
# ============================

# .env 파일 경로 (crawlers 폴더에 있음)
PROJECT_ROOT = Path('.').resolve().parent  # notebooks/ 에서 실행 시
env_path = PROJECT_ROOT / 'crawlers' / '.env'
if not env_path.exists():
    env_path = Path('.') / '.env'  # 현재 폴더
if not env_path.exists():
    env_path = Path('..') / 'crawlers' / '.env'

load_dotenv(env_path)
print(f'.env 로드: {env_path}')

NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID', '')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY', '')

print(f'네이버 API: {"OK" if NAVER_CLIENT_ID else "미설정"}')
print(f'유튜브 API: {"OK" if YOUTUBE_API_KEY else "미설정"}')

# T맵 데이터 경로
TMAP_DIR = Path(r'C:\Users\HP\Desktop\01.데이터\04. 내비게이션 데이터')
OUTPUT_DIR = Path(r'C:\Users\HP\Desktop\02.분석결과')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'T맵: {"OK" if TMAP_DIR.exists() else "경로 확인"}  ({TMAP_DIR})')
print(f'결과: {OUTPUT_DIR}')

In [ ]:
# ============================
# 방송 이벤트 정의
# ============================
BROADCASTS = [
    {"name": "전국노래자랑", "air_date": "2025-06-08", "air_end": "2025-06-08",
     "rating": 6.5, "budget": 3518, "genre": "음악예능",
     "spots": ["신정호"], "keywords": ["전국노래자랑 아산", "신정호정원"]},
    {"name": "전현무계획2", "air_date": "2025-11-07", "air_end": "2025-11-07",
     "rating": 1.5, "budget": 50000, "genre": "여행예능",
     "spots": [], "keywords": ["전현무계획 아산", "전현무계획2"]},
    {"name": "굿모닝대한민국", "air_date": "2025-11-12", "air_end": "2025-11-16",
     "rating": 0.55, "budget": 20000, "genre": "교양",
     "spots": ["온양온천", "곡교천", "현충사", "피나클랜드"],
     "keywords": ["굿모닝대한민국 아산", "온양온천", "곡교천 은행나무길"]},
    {"name": "6시내고향", "air_date": "2025-11-13", "air_end": "2025-11-13",
     "rating": 5.5, "budget": 110000, "genre": "교양",
     "spots": [], "keywords": ["6시내고향 아산"]},
    {"name": "같이삽시다", "air_date": "2025-11-24", "air_end": "2025-12-15",
     "rating": 3.0, "budget": 133000, "genre": "예능",
     "spots": ["곡교천", "신정호", "외암민속마을", "온양온천", "도고온천", "영인산"],
     "keywords": ["같이삽시다 아산", "박원숙 아산", "외암민속마을"]},
    {"name": "뛰어야산다2", "air_date": "2026-01-12", "air_end": "2026-01-12",
     "rating": 1.5, "budget": 45000, "genre": "스포츠예능",
     "spots": ["신정호", "곡교천", "현충사"],
     "keywords": ["뛰어야산다 아산", "뛰어야산다2"]},
    {"name": "황제파워", "air_date": "2026-05-09", "air_end": "2026-05-09",
     "rating": None, "budget": 220000, "genre": "라디오",
     "spots": ["온양온천"],
     "keywords": ["황제성 황제파워 아산"]},
]

COLORS = ['#2196F3','#FF5722','#4CAF50','#FFC107','#9C27B0','#00BCD4','#E91E63']
print(f'방송 {len(BROADCASTS)}건 정의')

---
## 1. 네이버 데이터랩 - 검색 트렌드

In [ ]:
# 네이버 데이터랩 API 함수
def naver_datalab_trend(keywords, start_date, end_date, time_unit='date'):
    """네이버 검색 트렌드 조회. keywords: list (최대5개)"""
    url = 'https://openapi.naver.com/v1/datalab/search'
    headers = {
        'X-Naver-Client-Id': NAVER_CLIENT_ID,
        'X-Naver-Client-Secret': NAVER_CLIENT_SECRET,
        'Content-Type': 'application/json',
    }
    body = {
        'startDate': start_date,
        'endDate': end_date,
        'timeUnit': time_unit,
        'keywordGroups': [{'groupName': kw, 'keywords': [kw]} for kw in keywords[:5]],
    }
    resp = httpx.post(url, json=body, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    
    rows = []
    for result in data.get('results', []):
        kw = result.get('title', '')
        for pt in result.get('data', []):
            rows.append({
                'keyword': kw,
                'date': pt['period'],
                'ratio': pt['ratio'],
            })
    return pd.DataFrame(rows)

print('네이버 데이터랩 함수 정의 완료')

In [ ]:
# 방송별 검색 트렌드 수집
# 핵심 키워드: 방송 전 30일 ~ 후 30일

all_trends = []

for b in tqdm(BROADCASTS, desc='검색 트렌드 수집'):
    air = datetime.strptime(b['air_date'], '%Y-%m-%d')
    start = (air - timedelta(days=30)).strftime('%Y-%m-%d')
    end = min(air + timedelta(days=30), datetime.now()).strftime('%Y-%m-%d')
    
    # 방송명 + 아산여행 + 해당 관광지
    kws = b['keywords'][:3] + ['아산 여행', '아산 관광']
    kws = list(dict.fromkeys(kws))[:5]  # 중복제거, 최대5개
    
    try:
        df = naver_datalab_trend(kws, start, end)
        df['broadcast'] = b['name']
        df['air_date'] = b['air_date']
        all_trends.append(df)
        print(f"  {b['name']}: {len(df)}행 ({start} ~ {end})")
    except Exception as e:
        print(f"  {b['name']}: 실패 - {e}")
    time.sleep(0.3)  # rate limit

if all_trends:
    df_trends = pd.concat(all_trends, ignore_index=True)
    df_trends['date'] = pd.to_datetime(df_trends['date'])
    df_trends.to_csv(OUTPUT_DIR / 'naver_search_trends.csv', index=False, encoding='utf-8-sig')
    print(f'\n전체: {len(df_trends):,}행 저장')
else:
    df_trends = pd.DataFrame()
    print('트렌드 데이터 없음')

In [ ]:
# 검색 트렌드 시각화 - 방송별
if len(df_trends) > 0:
    broadcasts_with_data = df_trends['broadcast'].unique()
    n = len(broadcasts_with_data)
    fig, axes = plt.subplots(n, 1, figsize=(16, 4*n))
    if n == 1: axes = [axes]

    for idx, bname in enumerate(broadcasts_with_data):
        ax = axes[idx]
        bdf = df_trends[df_trends['broadcast'] == bname]
        air_date = pd.Timestamp(bdf['air_date'].iloc[0])

        for i, kw in enumerate(bdf['keyword'].unique()):
            kdf = bdf[bdf['keyword'] == kw].sort_values('date')
            ax.plot(kdf['date'], kdf['ratio'], label=kw, color=COLORS[i % len(COLORS)], linewidth=1.5)

        ax.axvline(x=air_date, color='red', linestyle='--', linewidth=2, label='방영일')
        ax.fill_betweenx(ax.get_ylim(), air_date, air_date + timedelta(days=7),
                         alpha=0.1, color='red')
        ax.set_title(f'{bname} ({air_date.strftime("%Y-%m-%d")})', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(True, alpha=0.3)
        ax.set_ylabel('검색량 (상대값)')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_search_trends_by_broadcast.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 검색 트렌드 전후 비교 요약
if len(df_trends) > 0:
    summary_rows = []
    for bname in df_trends['broadcast'].unique():
        bdf = df_trends[df_trends['broadcast'] == bname]
        air = pd.Timestamp(bdf['air_date'].iloc[0])
        
        for kw in bdf['keyword'].unique():
            kdf = bdf[bdf['keyword'] == kw]
            before = kdf[kdf['date'] < air]['ratio']
            after = kdf[kdf['date'] >= air]['ratio']
            
            before_avg = before.mean() if len(before) > 0 else 0
            after_avg = after.mean() if len(after) > 0 else 0
            peak = kdf['ratio'].max()
            peak_date = kdf.loc[kdf['ratio'].idxmax(), 'date'] if len(kdf) > 0 else None
            lift = ((after_avg - before_avg) / before_avg * 100) if before_avg > 0 else None
            
            summary_rows.append({
                '방송': bname, '키워드': kw,
                '방송전_평균': round(before_avg, 1),
                '방송후_평균': round(after_avg, 1),
                '피크값': round(peak, 1),
                '피크일': str(peak_date)[:10] if peak_date is not None else '',
                '변화율(%)': round(lift, 1) if lift is not None else None,
            })
    
    df_trend_summary = pd.DataFrame(summary_rows)
    df_trend_summary.to_csv(OUTPUT_DIR / 'search_trend_summary.csv', index=False, encoding='utf-8-sig')
    display(df_trend_summary)

---
## 2. 네이버 블로그/뉴스 - 콘텐츠 수

In [ ]:
# 네이버 검색 API
def naver_search(query, search_type='blog', display=100, start=1, sort='date'):
    """search_type: blog, news, cafearticle"""
    url = f'https://openapi.naver.com/v1/search/{search_type}.json'
    headers = {
        'X-Naver-Client-Id': NAVER_CLIENT_ID,
        'X-Naver-Client-Secret': NAVER_CLIENT_SECRET,
    }
    params = {'query': query, 'display': display, 'start': start, 'sort': sort}
    resp = httpx.get(url, params=params, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

def collect_naver_content(keyword, search_type='blog', max_items=200):
    """네이버 블로그/뉴스 수집"""
    items = []
    for start in range(1, min(max_items, 1000), 100):
        try:
            data = naver_search(keyword, search_type, display=100, start=start)
            batch = data.get('items', [])
            if not batch: break
            for it in batch:
                title = re.sub(r'<[^>]+>', '', it.get('title', ''))
                desc = re.sub(r'<[^>]+>', '', it.get('description', ''))
                date_str = it.get('postdate', it.get('pubDate', ''))
                # 날짜 파싱
                pub_date = None
                if date_str and len(date_str) == 8 and date_str.isdigit():
                    pub_date = datetime.strptime(date_str, '%Y%m%d')
                elif date_str:
                    try:
                        from email.utils import parsedate_to_datetime
                        pub_date = parsedate_to_datetime(date_str)
                    except: pass
                items.append({
                    'keyword': keyword,
                    'type': search_type,
                    'title': title,
                    'description': desc[:200],
                    'url': it.get('link', ''),
                    'author': it.get('bloggername', it.get('cafename', '')),
                    'date': pub_date,
                })
            time.sleep(0.1)
        except Exception as e:
            print(f'  에러: {e}')
            break
    return items

print('네이버 검색 함수 정의 완료')

In [ ]:
# 방송별 블로그/뉴스 수집
all_content = []

# 수집 키워드 (핵심만)
crawl_keywords = [
    '아산 여행', '아산 관광', '아산 맛집',
    '신정호정원', '외암민속마을', '온양온천', '곡교천 은행나무길', '현충사 아산',
    '전국노래자랑 아산', '전현무계획 아산', '같이삽시다 아산', '뛰어야산다 아산',
    '6시내고향 아산', '굿모닝대한민국 아산',
]

for kw in tqdm(crawl_keywords, desc='블로그/뉴스 수집'):
    for stype in ['blog', 'news']:
        items = collect_naver_content(kw, stype, max_items=200)
        all_content.extend(items)
        print(f'  [{stype}] {kw}: {len(items)}건')

if all_content:
    df_content = pd.DataFrame(all_content)
    df_content['date'] = pd.to_datetime(df_content['date'], errors='coerce')
    df_content.to_csv(OUTPUT_DIR / 'naver_content.csv', index=False, encoding='utf-8-sig')
    print(f'\n전체: {len(df_content):,}건 저장')
    print(f'기간: {df_content["date"].min()} ~ {df_content["date"].max()}')
else:
    df_content = pd.DataFrame()

In [ ]:
# 블로그 게시물 수 일별 추이 + 방송 이벤트
if len(df_content) > 0:
    blog = df_content[df_content['type'] == 'blog'].copy()
    blog = blog[blog['date'] >= '2025-01-01']  # 2025년~
    
    if len(blog) > 0:
        daily_posts = blog.groupby(blog['date'].dt.date).size().reset_index()
        daily_posts.columns = ['date', 'post_count']
        daily_posts['date'] = pd.to_datetime(daily_posts['date'])
        
        fig, ax = plt.subplots(figsize=(16, 5))
        ax.bar(daily_posts['date'], daily_posts['post_count'], color='#2196F3', alpha=0.6, width=1)
        ma = daily_posts['post_count'].rolling(7, min_periods=1).mean()
        ax.plot(daily_posts['date'], ma, color='#FF5722', linewidth=2, label='7일 이동평균')
        
        ymax = daily_posts['post_count'].max() * 1.1
        for i, b in enumerate(BROADCASTS):
            air = pd.Timestamp(b['air_date'])
            c = COLORS[i % len(COLORS)]
            ax.axvline(x=air, color=c, linestyle='--', alpha=0.7)
            ax.text(air, ymax * (0.95 - i*0.07), f" {b['name']}", fontsize=7, color=c, va='top')
        
        ax.set_title('아산 관련 네이버 블로그 게시물 수 + 방송 이벤트', fontsize=14, fontweight='bold')
        ax.set_ylabel('일별 게시물 수')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'fig_blog_timeline.png', dpi=150, bbox_inches='tight')
        plt.show()

---
## 3. 유튜브 영상 수집

In [ ]:
def youtube_search(query, start_date, end_date, max_results=50):
    """YouTube 검색 + 통계"""
    search_url = 'https://www.googleapis.com/youtube/v3/search'
    videos_url = 'https://www.googleapis.com/youtube/v3/videos'
    
    params = {
        'part': 'snippet', 'q': query, 'type': 'video', 'order': 'date',
        'publishedAfter': f'{start_date}T00:00:00Z',
        'publishedBefore': f'{end_date}T23:59:59Z',
        'maxResults': min(max_results, 50),
        'key': YOUTUBE_API_KEY, 'regionCode': 'KR', 'relevanceLanguage': 'ko',
    }
    resp = httpx.get(search_url, params=params, timeout=30)
    resp.raise_for_status()
    search_data = resp.json()
    
    items = search_data.get('items', [])
    if not items: return []
    
    # 통계 가져오기
    vids = [it['id']['videoId'] for it in items if 'videoId' in it.get('id', {})]
    stats = {}
    if vids:
        sr = httpx.get(videos_url, params={'part': 'statistics', 'id': ','.join(vids), 'key': YOUTUBE_API_KEY}, timeout=30)
        for it in sr.json().get('items', []):
            stats[it['id']] = it.get('statistics', {})
    
    results = []
    for it in items:
        vid = it.get('id', {}).get('videoId', '')
        snip = it.get('snippet', {})
        st = stats.get(vid, {})
        results.append({
            'keyword': query,
            'title': snip.get('title', ''),
            'channel': snip.get('channelTitle', ''),
            'published': snip.get('publishedAt', '')[:10],
            'url': f'https://www.youtube.com/watch?v={vid}',
            'views': int(st.get('viewCount', 0)),
            'likes': int(st.get('likeCount', 0)),
            'comments': int(st.get('commentCount', 0)),
        })
    return results

print('유튜브 함수 정의 완료')

In [ ]:
# 유튜브 수집
yt_keywords = [
    '아산 여행', '아산 관광', '온양온천', '외암민속마을',
    '전국노래자랑 아산', '같이삽시다 아산', '전현무계획 아산', '뛰어야산다 아산',
]

all_yt = []
for kw in tqdm(yt_keywords, desc='유튜브 수집'):
    try:
        items = youtube_search(kw, '2025-01-01', datetime.now().strftime('%Y-%m-%d'), max_results=50)
        all_yt.extend(items)
        print(f'  {kw}: {len(items)}건')
    except Exception as e:
        print(f'  {kw}: 실패 - {e}')
    time.sleep(0.2)

if all_yt:
    df_yt = pd.DataFrame(all_yt)
    df_yt['published'] = pd.to_datetime(df_yt['published'])
    df_yt = df_yt.drop_duplicates(subset='url')
    df_yt.to_csv(OUTPUT_DIR / 'youtube_videos.csv', index=False, encoding='utf-8-sig')
    print(f'\n전체: {len(df_yt):,}건 (중복 제거 후)')
    print(f'총 조회수: {df_yt["views"].sum():,}')
else:
    df_yt = pd.DataFrame()

In [ ]:
# 유튜브 업로드 추이 + 방송 이벤트
if len(df_yt) > 0:
    yt_daily = df_yt.groupby('published').agg(
        videos=('url', 'count'), total_views=('views', 'sum')
    ).reset_index()
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    
    axes[0].bar(yt_daily['published'], yt_daily['videos'], color='#F44336', alpha=0.6)
    axes[0].set_title('아산 관련 유튜브 영상 업로드 수', fontweight='bold')
    axes[0].set_ylabel('영상 수')
    
    axes[1].bar(yt_daily['published'], yt_daily['total_views'], color='#2196F3', alpha=0.6)
    axes[1].set_title('일별 총 조회수', fontweight='bold')
    axes[1].set_ylabel('조회수')
    
    for ax in axes:
        for i, b in enumerate(BROADCASTS):
            air = pd.Timestamp(b['air_date'])
            ax.axvline(x=air, color=COLORS[i%len(COLORS)], linestyle='--', alpha=0.7)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_youtube_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 4. T맵 관광지 방문 분석

In [ ]:
# T맵 데이터 로드
POI_KEYWORDS = {
    '신정호': ['신정호', '신정호정원'],
    '현충사': ['현충사'],
    '외암민속마을': ['외암', '외암민속'],
    '곡교천': ['곡교천', '은행나무길'],
    '온양온천': ['온양온천', '온천', '족욕'],
    '도고온천': ['도고', '도고파라다이스'],
    '피나클랜드': ['피나클랜드'],
}

def match_poi(name):
    if pd.isna(name): return None
    s = str(name).lower()
    for poi, kws in POI_KEYWORDS.items():
        for kw in kws:
            if kw in s: return poi
    return None

tmap_files = sorted(glob.glob(str(TMAP_DIR / '**' / 'as_tmap_od_*.csv'), recursive=True))
print(f'T맵 파일: {len(tmap_files)}개')

chunks = []
for f in tqdm(tmap_files, desc='T맵 로드'):
    for enc in ['utf-8-sig', 'cp949']:
        try:
            chunks.append(pd.read_csv(f, encoding=enc))
            break
        except: continue

df_tmap = pd.concat(chunks, ignore_index=True)
df_tmap['drv_ymd'] = pd.to_datetime(df_tmap['drv_ymd'].astype(str))
df_tmap['matched_poi'] = df_tmap['dstn_nm'].apply(match_poi)
df_tmap['is_tourism'] = df_tmap['dstn_ctgy'].str.startswith('여행/레저').fillna(False)

print(f'전체: {len(df_tmap):,}행, {df_tmap["drv_ymd"].min()} ~ {df_tmap["drv_ymd"].max()}')
print(f'관광지 매칭: {df_tmap["matched_poi"].notna().sum():,}행')

In [ ]:
# T맵 관광지별 방문 시계열 (2024~만)
poi_daily = df_tmap[df_tmap['matched_poi'].notna()].groupby(
    ['drv_ymd', 'matched_poi']
).agg(visits=('vst_cnt', 'sum'), avg_stay=('avg_stay_min', 'mean')).reset_index()

poi_viz = poi_daily[poi_daily['drv_ymd'] >= '2024-06-01'].copy()
pois = poi_viz['matched_poi'].value_counts().head(6).index.tolist()

fig, axes = plt.subplots(len(pois), 1, figsize=(16, 3*len(pois)), sharex=True)
if len(pois) == 1: axes = [axes]

for idx, poi in enumerate(pois):
    ax = axes[idx]
    ts = poi_viz[poi_viz['matched_poi'] == poi].sort_values('drv_ymd')
    ax.plot(ts['drv_ymd'], ts['visits'], alpha=0.3, color='#2196F3', linewidth=0.8)
    if len(ts) > 7:
        ax.plot(ts['drv_ymd'], ts['visits'].rolling(7, min_periods=1).mean(),
                color='#FF5722', linewidth=2)
    # 방송 표시 (해당 관광지가 노출된 방송만)
    for b in BROADCASTS:
        if poi in b['spots'] or not b['spots']:
            air = pd.Timestamp(b['air_date'])
            if ts['drv_ymd'].min() <= air <= ts['drv_ymd'].max() + timedelta(days=30):
                ax.axvline(x=air, color='red', linestyle='--', alpha=0.7)
                ax.text(air, ax.get_ylim()[1]*0.9, f' {b["name"][:6]}',
                        fontsize=7, color='red', rotation=90, va='top')
    ax.set_title(poi, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_tmap_poi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# T맵 방송 전후 비교
WINDOW = 28  # 4주
tmap_results = []

for b in BROADCASTS:
    air = pd.Timestamp(b['air_date'])
    air_end = pd.Timestamp(b['air_end'])
    
    # 해당 관광지 또는 전체 관광
    if b['spots']:
        mask = df_tmap['matched_poi'].isin(b['spots'])
    else:
        mask = df_tmap['is_tourism']
    
    pre = df_tmap[mask & (df_tmap['drv_ymd'] >= air - timedelta(days=WINDOW)) & (df_tmap['drv_ymd'] < air)]['vst_cnt'].sum()
    post = df_tmap[mask & (df_tmap['drv_ymd'] > air_end) & (df_tmap['drv_ymd'] <= air_end + timedelta(days=WINDOW))]['vst_cnt'].sum()
    
    pre_avg = pre / WINDOW
    post_avg = post / WINDOW
    chg = (post_avg - pre_avg) / pre_avg * 100 if pre_avg > 0 else None
    
    # 전년 동기
    yoy_air = air - timedelta(days=365)
    yoy = df_tmap[mask & (df_tmap['drv_ymd'] >= yoy_air) & (df_tmap['drv_ymd'] <= yoy_air + timedelta(days=WINDOW))]['vst_cnt'].sum()
    yoy_avg = yoy / WINDOW
    
    tmap_results.append({
        '방송': b['name'], '방영일': b['air_date'],
        '관광지': '+'.join(b['spots'][:3]) or '전체',
        '방송전_일평균': round(pre_avg, 1),
        '방송후_일평균': round(post_avg, 1),
        '변화율(%)': round(chg, 1) if chg else None,
        '전년동기_일평균': round(yoy_avg, 1),
    })

df_tmap_results = pd.DataFrame(tmap_results)
df_tmap_results.to_csv(OUTPUT_DIR / 'tmap_broadcast_comparison.csv', index=False, encoding='utf-8-sig')
display(df_tmap_results)

---
## 5. 종합 대시보드

In [ ]:
# 종합 비교 차트
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

broadcast_names = [b['name'] for b in BROADCASTS]
x = range(len(broadcast_names))

# 1. T맵 방문 전후
if len(df_tmap_results) > 0:
    ax = axes[0]
    w = 0.35
    ax.bar([i-w/2 for i in x], df_tmap_results['방송전_일평균'], w, label='방송 전', color='#90CAF9')
    ax.bar([i+w/2 for i in x], df_tmap_results['방송후_일평균'], w, label='방송 후', color='#FF8A65')
    ax.set_xticks(list(x))
    ax.set_xticklabels(broadcast_names, rotation=45, ha='right', fontsize=8)
    ax.set_title('T맵 관광지 방문', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# 2. 검색 트렌드 변화율 (아산 여행 키워드만)
if len(df_trends) > 0:
    ax = axes[1]
    travel_lifts = []
    for b in BROADCASTS:
        bdf = df_trends[df_trends['broadcast'] == b['name']]
        kdf = bdf[bdf['keyword'] == '아산 여행']
        if len(kdf) == 0:
            kdf = bdf[bdf['keyword'] == bdf['keyword'].iloc[0]] if len(bdf) > 0 else pd.DataFrame()
        if len(kdf) > 0:
            air = pd.Timestamp(b['air_date'])
            before_avg = kdf[kdf['date'] < air]['ratio'].mean()
            after_avg = kdf[kdf['date'] >= air]['ratio'].mean()
            lift = ((after_avg - before_avg) / before_avg * 100) if before_avg > 0 else 0
            travel_lifts.append(lift)
        else:
            travel_lifts.append(0)
    
    colors = ['#4CAF50' if v > 0 else '#F44336' for v in travel_lifts]
    ax.barh(broadcast_names, travel_lifts, color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_title('검색 트렌드 변화율 (%)', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# 3. 예산 vs 시청률
ax = axes[2]
budgets = [b['budget']/1000 for b in BROADCASTS]  # 백만원
ratings = [b['rating'] if b['rating'] else 0 for b in BROADCASTS]
ax.scatter(budgets, ratings, s=100, c=COLORS[:len(BROADCASTS)], zorder=5)
for i, b in enumerate(BROADCASTS):
    ax.annotate(b['name'], (budgets[i], ratings[i]), fontsize=7,
                xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('예산 (백만원)')
ax.set_ylabel('시청률 (%)')
ax.set_title('예산 vs 시청률', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 파일 목록
print('=' * 60)
print('생성된 파일')
print('=' * 60)
for f in sorted(OUTPUT_DIR.glob('*')):
    sz = f.stat().st_size / 1024
    print(f'  {f.name:45s} {sz:>8,.0f} KB')
print(f'\n총 {len(list(OUTPUT_DIR.glob("*")))}개')